# Product neighbors

These baskets are synthetic. They are not the Day 1 orders. Each basket is a few product names. Seed 7 draws 800 baskets.

You hold out 25 percent of the baskets. For each held-out basket, hide the last product name in alphabetical order and ask for k neighbors. Precision@k is 1/k when that product is in the list, else 0. The mean is the score.

The public notebook uses this same split. There is no hidden basket file.

Popularity recommends the most common train products. Cosine similarity recommends products that share baskets with the ones still in the query.


In [ ]:
import sys
from pathlib import Path

here = Path.cwd().resolve()
root = None
for candidate in [here, *here.parents]:
    marker = candidate / "lib" / "leaks.py"
    if marker.is_file() and (candidate / "app.py").is_file():
        root = candidate
        break
if root is None:
    raise FileNotFoundError("Start Jupyter from 02-segments-and-recommendations.")
lib = root / "lib"
if str(lib) not in sys.path:
    sys.path.insert(0, str(lib))
ROOT = root
print("module", ROOT)


In [ ]:
from baskets import (
    item_sets,
    item_similarity,
    make_baskets,
    neighbor_table,
    popularity,
    precision_at_k,
    recommend_popular,
    recommend_similar,
    split_baskets,
)

baskets = make_baskets()
train, held = split_baskets(baskets)
print(len(train), "train baskets", len(held), "held-out baskets")
print(baskets.head(3).to_string(index=False))


In [ ]:
k = None  # neighbors to return, try 3

if k is None:
    raise NotImplementedError("Set k.")

train_sets = item_sets(train)
test_sets = item_sets(held)
ranking = popularity(train_sets)
sim = item_similarity(train_sets)
popular_score = precision_at_k(test_sets, lambda query, k: recommend_popular(ranking, query, k), k)
similar_score = precision_at_k(test_sets, lambda query, k: recommend_similar(sim, query, k), k)
print("popularity precision", round(popular_score, 3))
print("cosine precision", round(similar_score, 3))


In [ ]:
neighbors = neighbor_table(sim, k)
out = ROOT / "models" / "neighbors.csv"
out.parent.mkdir(exist_ok=True)
neighbors.to_csv(out, index=False)
print(neighbors.head(6).to_string(index=False))
print("saved", out)


In [ ]:
winner = None  # "popularity" or "cosine"

if winner is None:
    raise NotImplementedError("Name the higher precision.")
print(winner)
